# A particle on a cone

A bead slides without friction on the inside of a cone whose tip is at the origin
and whose axis is the $z$ axis, opening upward, with half-angle $\alpha$. Gravity
points down. Will Raven's Ch 7 demo: one constraint, two generalized coordinates,
and the motion that comes out.

Cylindrical coordinates $(\rho, \phi, z)$ describe where the bead is, and the cone
is the constraint $z = \rho\cot\alpha$. So two numbers, $\rho$ and $\phi$, say where
the bead is:

$$x = \rho\cos\phi,\qquad y = \rho\sin\phi,\qquad z = \rho\cot\alpha .$$

The sliders are Will's: where the bead starts ($\rho(0)$, $\phi(0)$) and how it is
moving ($\dot\rho(0)$, $\dot\phi(0)$), then the cone's half-angle $\alpha$, gravity
on or off, and `t`, how far along the orbit the bead has got (the red dot). The
left picture is the cone in 3D; the right one looks down the $z$ axis.

## 1. Setup

The equations of motion for $\rho(t)$ and $\phi(t)$ are in `equations` below (where
they come from is section 4). The code stops the bead if it reaches the tip of the
cone, since the equations say nothing about what happens there.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets

t_max = 10.0            # s, how long to follow the bead

def equations(t, state, h, alpha, g):
    # state = [rho, rho_dot, phi]; h = rho^2 phi_dot is constant (section 4)
    rho, rho_dot, phi = state
    rho_ddot = -g * np.sin(alpha) * np.cos(alpha) + h**2 * np.sin(alpha)**2 / rho**3
    phi_dot = h / rho**2
    return [rho_dot, rho_ddot, phi_dot]

def at_the_tip(t, state, h, alpha, g):
    return state[0] - 0.01      # zero when rho reaches 1 cm: stop there
at_the_tip.terminal = True

def solve(rho0, rho_dot0, phi0, phi_dot0, alpha, g):
    h = rho0**2 * phi_dot0
    t = np.linspace(0, t_max, 2001)
    sol = solve_ivp(equations, (0, t_max), [rho0, rho_dot0, phi0], args=(h, alpha, g),
                    t_eval=t, events=at_the_tip, rtol=1e-8, atol=1e-10)
    return sol.t, sol.y[0], sol.y[2]        # t, rho(t), phi(t)

## 2. The orbit

In [ ]:
def cone_xyz(rho, phi, alpha):
    return rho * np.cos(phi), rho * np.sin(phi), rho / np.tan(alpha)

def draw(rho0=1.0, rho_dot0=0.0, phi0=0.0, phi_dot0=2.4, alpha_deg=45, g=9.8, t=10.0):
    alpha = np.radians(alpha_deg)
    time, rho, phi = solve(rho0, rho_dot0, phi0, phi_dot0, alpha, g)
    shown = time <= t
    x, y, z = cone_xyz(rho, phi, alpha)

    fig = plt.figure(figsize=(11, 5))
    ax3 = fig.add_subplot(1, 2, 1, projection='3d')
    ax2 = fig.add_subplot(1, 2, 2)

    # the cone, out to a little beyond the largest rho the bead reaches
    rho_max = 1.1 * max(rho.max(), rho0)
    r, p = np.meshgrid(np.linspace(0, rho_max, 15), np.linspace(0, 2 * np.pi, 60))
    cx, cy, cz = cone_xyz(r, p, alpha)
    ax3.plot_surface(cx, cy, cz, color='gray', alpha=0.2, edgecolor='none')

    # the path so far, where it started (black) and where it is now (red)
    ax3.plot(x[shown], y[shown], z[shown], color='tab:blue', lw=2)
    ax3.scatter(x[0], y[0], z[0], color='black', s=40)
    ax3.scatter(x[shown][-1], y[shown][-1], z[shown][-1], color='red', s=60)
    ax3.set_xlabel('x (m)'); ax3.set_ylabel('y (m)'); ax3.set_zlabel('z (m)')
    ax3.set_title(f'bead on a cone, t = {time[shown][-1]:.2f} s')
    ax3.view_init(elev=20, azim=-60)

    # seen from above
    ax2.plot(x[shown], y[shown], color='tab:blue', lw=2)
    ax2.scatter(x[0], y[0], color='black', s=40)
    ax2.scatter(x[shown][-1], y[shown][-1], color='red', s=60)
    ax2.add_patch(plt.Circle((0, 0), rho_max, fill=False, color='gray', ls='--'))
    ax2.set_aspect('equal')
    ax2.set_xlim(-rho_max, rho_max); ax2.set_ylim(-rho_max, rho_max)
    ax2.set_xlabel('x (m)'); ax2.set_ylabel('y (m)')
    ax2.set_title('seen from above')
    if time[-1] < t_max:
        fig.suptitle(f'the bead reached the tip at t = {time[-1]:.2f} s', color='red')
    plt.show()

def slider(value, lo, hi, step, label):
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step,
                               description=label, continuous_update=False)

widgets.interact(draw,
                 rho0=slider(1.0, 0.1, 5, 0.1, 'rho(0) m'),
                 rho_dot0=slider(0.0, -2, 2, 0.1, 'rho_dot(0)'),
                 phi0=slider(0.0, 0, 2 * np.pi, 0.1, 'phi(0)'),
                 phi_dot0=slider(2.4, -2, 4, 0.1, 'phi_dot(0)'),
                 alpha_deg=slider(45, 15, 75, 7.5, 'alpha (deg)'),
                 g=widgets.ToggleButtons(options=[('gravity on', 9.8), ('gravity off', 0.0)], description='g'),
                 t=slider(10.0, 0.01, t_max, 0.1, 't (s)'));

## 3. Interesting cases

- **Will's rosette:** the default, $\dot\phi(0) = 2.4$ rad/s. The bead swings between
  an outer turning point (where it started) and an inner one, and the pattern
  precesses: a rosette, the same shape that shows up for orbits that are not quite
  Kepler's. Drag `t` back to 0 and forward again to watch it go.
- **A circular orbit:** $\dot\phi(0) = 3.13$ rad/s. The bead stays at $\rho = 1$ m
  and just goes around. A little faster or slower and it wobbles about that circle.
  Why that value? Section 4.
- **Too slow:** $\dot\phi(0) = 0.5$. The inner turning point moves in toward the tip
  but never reaches it. Can the bead reach the tip at all if $\dot\phi(0) \neq 0$?
- **Dropped:** $\dot\phi(0) = 0$. Straight down the side to the tip; the integration
  stops there (red title).
- **No gravity:** press **gravity off**. The path is a straight line on the cone
  when the cone is unrolled flat; the bead comes in, swings around, and leaves for
  good.
- **A sharper or flatter cone:** move `alpha`. A flatter cone (larger $\alpha$) is
  closer to a tabletop; a sharper one is closer to a hole. The circular-orbit
  speed changes with it (section 4).
- **A radial kick:** `rho_dot(0)` = 0.5 with the circular orbit's `phi_dot(0)`.
  The orbit wobbles about the circle instead of running away: Taylor 7.38(c).

## 4. Where the equations come from

The velocity in cylindrical coordinates is $\dot{\mathbf r} = (\dot\rho,\ \rho\dot\phi,\ \dot z)$
and on the cone $\dot z = \dot\rho\cot\alpha$, so

$$\mathcal L = T - U
 = \tfrac12 m\left(\dot\rho^2 + \rho^2\dot\phi^2 + \dot\rho^2\cot^2\alpha\right) - m g\rho\cot\alpha
 = \tfrac12 m\left(\frac{\dot\rho^2}{\sin^2\alpha} + \rho^2\dot\phi^2\right) - m g\rho\cot\alpha .$$

The Lagrange equation for $\rho$ gives

$$\ddot\rho = \sin^2\!\alpha\,\rho\dot\phi^2 - g\sin\alpha\cos\alpha .$$

$\mathcal L$ does not contain $\phi$, so $\partial\mathcal L/\partial\dot\phi = m\rho^2\dot\phi$
is constant: angular momentum about the $z$ axis is conserved. Writing it as
$m h$ with $h = \rho_0^2\dot\phi(0)$ gives $\dot\phi = h/\rho^2$, and the code
integrates these two equations.

The circular orbit has $\ddot\rho = 0$ at $\rho = \rho_0$:
$\rho_0\dot\phi^2\sin^2\alpha = g\sin\alpha\cos\alpha$, so
$\dot\phi(0) = \sqrt{g\cot\alpha/\rho_0}$, which is $\sqrt{9.8} = 3.13$ rad/s
for $\alpha = 45^\circ$ and $\rho_0 = 1$ m.

## 5. What you found